# Baseline 2 – Truy xuất bằng Perceptual Hash (pHash)

**Mục tiêu:** Triển khai truy xuất ảnh tương đồng Shopee bằng cột `image_phash` và khoảng cách **Hamming** trên **toàn bộ 34,250 dòng** của train.csv.


## 1. Import thư viện

Cell code sau thiết lập môi trường và đường dẫn dữ liệu.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

DATA_DIR  = '../data/raw/'
CSV_PATH  = os.path.join(DATA_DIR, 'train.csv')
PROCESSED = '../data/processed/'
RESULTS   = '../results/'
os.makedirs(RESULTS, exist_ok=True)

print('Import thư viện thành công!')


**Nhận xét:** pHash là baseline cổ điển, **không cần GPU**, phù hợp so sánh với deep learning.


## 2. Đọc toàn bộ dữ liệu train.csv (34,250 dòng)

- Đọc **toàn bộ** `train.csv` — tuyệt đối không dùng `.head()` hay cắt nhỏ dữ liệu.
- Mỗi query đều có đáp án trong gallery để đảm bảo giao thức đánh giá chính xác.


In [ ]:
# Đọc TOÀN BỘ train.csv
df = pd.read_csv(CSV_PATH)
print(f'Tổng số dòng: {len(df):,}')
print(f'Số nhóm (label_group): {df["label_group"].nunique():,}')
print(df.head(3))


**Nhận xét:** Ground truth = cùng `label_group` (cùng sản phẩm trong bài toán Price Match Guarantee).


## 3. Tính ma trận khoảng cách Hamming (vector hóa)

- `image_phash`: chuỗi hex 16 ký tự (64 bit).
- **Hamming distance** = số bit khác nhau (`XOR` rồi đếm bit 1).
- Khoảng cách **nhỏ** → ảnh **giống** hơn.
- Dùng NumPy vectorization để tính nhanh: chuyển hex → uint64, XOR theo cặp, đếm bit.
- Đặt đường chéo = +inf để **loại chính query** khỏi kết quả Top-K khi xếp hạng.


In [ ]:
def hex_to_uint64(hex_arr):
    """Chuyển mảng chuỗi hex sang mảng uint64."""
    return np.array([int(h, 16) for h in hex_arr], dtype=np.uint64)

def popcount_uint64(x):
    """Đếm số bit 1 trong uint64 (vectorized)."""
    # Thuật toán popcount 64-bit
    x = x - ((x >> np.uint64(1)) & np.uint64(0x5555555555555555))
    x = (x & np.uint64(0x3333333333333333)) + ((x >> np.uint64(2)) & np.uint64(0x3333333333333333))
    x = (x + (x >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
    return ((x * np.uint64(0x0101010101010101)) >> np.uint64(56)).astype(np.int32)

phashes_hex = df['image_phash'].astype(str).values
n = len(phashes_hex)
phashes_uint = hex_to_uint64(phashes_hex)

print(f'Số ảnh: {n:,}')
print('Đang tính ma trận Hamming (batch)...')

# Tính theo batch để tiết kiệm RAM
BATCH = 1000
# Dùng int16 để tiết kiệm bộ nhớ (khoảng cách tối đa = 64)
dist_matrix = np.empty((n, n), dtype=np.int16)

for i in range(0, n, BATCH):
    end_i = min(i + BATCH, n)
    # XOR batch_i với toàn bộ phashes
    xor = np.bitwise_xor(phashes_uint[i:end_i, None], phashes_uint[None, :])  # shape (BATCH, n)
    dist_matrix[i:end_i, :] = popcount_uint64(xor)
    if (i // BATCH) % 5 == 0:
        print(f'  Đã xử lý {end_i:,}/{n:,} hàng...')

print(f'Kích thước ma trận: {dist_matrix.shape}')
print(f'Ví dụ Hamming query[0] → gallery[1]: {dist_matrix[0, 1]} bit')


**Nhận xét:** Ma trận Hamming 34K×34K tính bằng vectorization NumPy. pHash bỏ qua biến dạng nhẹ nhưng kém robust khi watermark / chỉnh màu mạnh.


## 4. Demo Top-5 truy xuất

Minh họa một query: xếp hạng theo Hamming tăng dần (đã loại chính nó).


In [ ]:
query_idx = 0
# Loại chính query bằng cách gán đường chéo = 999 tạm thời cho demo
dist_row = dist_matrix[query_idx].copy().astype(np.int32)
dist_row[query_idx] = 999  # loại self
top5_idx = np.argsort(dist_row)[:5]

q_group = df['label_group'].iloc[query_idx]

print(f'Query: {df["image"].iloc[query_idx]} | nhóm {q_group}')
print('Top-5 (Hamming nhỏ hơn = tốt hơn):')
for rank, j in enumerate(top5_idx, 1):
    dung = df['label_group'].iloc[j] == q_group
    print(
        f'  {rank}. d={dist_matrix[query_idx, j]:2d}  '
        f'{"✓ Đúng nhóm" if dung else "✗ Sai nhóm"}  '
        f'{df["image"].iloc[j]}'
    )


## 5. Precision@K, Recall@K và mAP

Tính metric với K ∈ {1, 3, 5, 10}; **BẮT BUỘC loại chính query** trước khi xếp hạng.


In [ ]:
K_LIST = [1, 3, 5, 10]
MAX_K  = max(K_LIST)
labels = df['label_group'].values
n = len(labels)

def average_precision(ranked_labels, true_label, total_relevant):
    if total_relevant == 0:
        return 0.0
    ap, hits = 0.0, 0
    for rank, label in enumerate(ranked_labels, 1):
        if label == true_label:
            hits += 1
            ap += hits / rank
    return ap / total_relevant

rows, all_ap = [], []
for i in range(n):
    true_label = labels[i]
    total_relevant = sum(1 for j in range(n) if labels[j] == true_label and j != i)
    if total_relevant == 0:
        continue

    # Lấy khoảng cách, gán self = 999 để loại query gốc
    dist_row = dist_matrix[i].copy().astype(np.int32)
    dist_row[i] = 999  # loại chính query

    top_idx = np.argsort(dist_row)[:MAX_K]  # Hamming nhỏ = gần = tốt
    ranked_labels = [labels[j] for j in top_idx]

    ap = average_precision(ranked_labels, true_label, total_relevant)
    all_ap.append(ap)

    row = {'posting_id': df['posting_id'].iloc[i], 'label_group': true_label}
    for k in K_LIST:
        top_k_labels = ranked_labels[:k]
        hits = sum(1 for lbl in top_k_labels if lbl == true_label)
        row[f'Precision@{k}'] = round(hits / k, 4)
        denom = min(total_relevant, k) if total_relevant > 0 else 1
        row[f'Recall@{k}'] = round(hits / denom, 4)
    rows.append(row)

detail_df = pd.DataFrame(rows)
detail_df['AP'] = [round(v, 4) for v in all_ap]

summary = []
for k in K_LIST:
    summary.append({
        'K': k,
        'Precision@K': round(detail_df[f'Precision@{k}'].mean(), 4),
        'Recall@K': round(detail_df[f'Recall@{k}'].mean(), 4),
    })
metrics_df = pd.DataFrame(summary)
mAP = round(float(np.mean(all_ap)), 4)

print(f'METRIC TRUNG BÌNH TRÊN TOÀN BỘ {n:,} ẢNH')
print(metrics_df.to_string(index=False))
print(f'\nmAP (mean Average Precision) = {mAP:.4f}')
metrics_df['mAP'] = mAP

metrics_df.to_csv(os.path.join(RESULTS, 'avg_metrics_phash.csv'), index=False)
detail_df.to_csv(os.path.join(RESULTS, 'detail_metrics_phash.csv'), index=False)
print(f'Đã lưu CSV vào {RESULTS}')


**Nhận xét:** mAP phản ánh mức hash có tách được ảnh cùng sản phẩm hay không — thường thấp hơn ResNet50 trên ảnh thương mại điện tử.


## 6. Biểu đồ metric


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(metrics_df['K'].astype(str), metrics_df['Precision@K'], color='steelblue')
axes[0].set_title('Precision@K – pHash')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Precision')
axes[0].set_ylim(0, 1)
for bar, val in zip(axes[0].patches, metrics_df['Precision@K']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

axes[1].bar(metrics_df['K'].astype(str), metrics_df['Recall@K'], color='coral')
axes[1].set_title('Recall@K – pHash')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Recall')
axes[1].set_ylim(0, 1)
for bar, val in zip(axes[1].patches, metrics_df['Recall@K']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

fig.suptitle(f'pHash – mAP = {mAP:.4f}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, 'metrics_phash.png'), dpi=150)
plt.show()


## 7. Xuất kết quả dự đoán Top-5

- Xuất file `ket_qua_phash.csv` gồm 2 cột: `posting_id` và `preds_phash` (list Top-5 posting_id được dự đoán).
- **BẮT BUỘC** loại chính query khỏi danh sách kết quả trước khi lấy Top-5.


In [ ]:
TOP_K_EXPORT = 5
posting_ids = df['posting_id'].values

result_rows = []
for i in range(n):
    # Gán self = 999 để loại chính query
    dist_row = dist_matrix[i].copy().astype(np.int32)
    dist_row[i] = 999  # loại chính query

    top5_idx = np.argsort(dist_row)[:TOP_K_EXPORT]  # Hamming nhỏ = gần = tốt
    preds = [posting_ids[j] for j in top5_idx]
    result_rows.append({
        'posting_id': posting_ids[i],
        'preds_phash': ' '.join(preds)
    })

result_df = pd.DataFrame(result_rows)

output_path = os.path.join(RESULTS, 'ket_qua_phash.csv')
result_df.to_csv(output_path, index=False)
print(f'Đã xuất kết quả: {output_path}')
print(f'Số dòng: {len(result_df):,}')
print('Mẫu 5 dòng đầu:')
print(result_df.head())
